# LLM-RecFusion — Stage 2: Qwen Dual-Tower 语义召回 (MovieLens-1M 真实数据)

**目标**:
1. 使用 `movies.dat` 构建 Item Tower：对所有 3,883 部电影提取语义 Embedding
2. 使用共享 Train Set 快照构建 User Tower（严格使用历史记忆，无数据泄露）
3. 基于 FAISS 向量检索，为全部 ~6040 个测试用户召回 Top-50 语义相似电影
4. 保存召回结果为 `data/results/llm_recall_ml_1m.json`

**架构要点 (Qwen 双塔 — 结构化语义表达)**：

**Item Tower (物品静态属性编码)**：
- 输入格式：`简短任务指令前缀 + 物品Key-Value结构化属性文本`
- 不拼接 ID 类特征，避免引入无效噪声
- 核心任务：完成物品静态属性的结构化语义表达

**User Tower (用户时序意图建模)**：
- 文本拼接顺序固定为：`用户历史记录 + 当前搜索内容`，保证搜索意图更高权重
- 若拼接后文本超出模型最大上下文上限，从左侧截断，优先保留近期行为与当前意图
- 重点关注时序信息与用户实时搜索意图的权重优先级

**对齐策略**：
- User/Item 两侧统一使用最后一个 token 的隐状态 (Last Token Pooling) 作为输出 Embedding

**全局一致的数据流**:
```
data/snapshots/ml_1m_train.parquet (与 ALS 笔记本共享的 Train Set 快照)
data/snapshots/ml_1m_test.parquet  (与 ALS 笔记本共享的 Test Set 快照)
data/snapshots/ml_1m_movies.parquet (电影元数据)
  → Item Tower: [任务指令] + title=X genre=Y → Qwen → Last Token Embedding → FAISS
  → User Tower: [用户历史] + [当前搜索] → Qwen → Last Token Embedding → FAISS
  → FAISS 批量检索所有 ~6040 个测试用户 → Top-50 召回 → 保存
```

**核心改进**:
- Batch 推理: 使用 `padding=True` (动态 Padding)，Batch Size=32，而非逐条推理
- 全量评估: 全部 ~6040 个用户，而非原来的 82 个
- 数据对齐: 与 ALS 笔记本共享完全相同的 Train/Test 划分
- Last Token Pooling: 代替 Mean Pooling，对齐最后一位置为输出 Embedding
- 左截断策略: 超长输入从左侧丢弃，优先保留近期行为与当前意图

In [5]:
# ============================================================
# Cell 1: 导入 & 全局配置
# ============================================================
import os
# [关键修复] 必须在 import torch 之前设置 PYTORCH_JIT=0,
# 否则 Qwen2.5 的 @torch.jit.script 类型注解会触发 PyTorch 旧版
# JIT schema 解析器的版本兼容 Bug (infer_schema 报错).
# 设置后请重启 Kernel 使本行生效！
os.environ["PYTORCH_JIT"] = "0"
import sys
import json
import warnings
from pathlib import Path

# [修复] 将项目根目录加入 sys.path，确保能 import pipeline/ 和 models/
#     向上遍历目录树，找到同时包含 pipeline/ 和 notebooks/ 的目录
_proj_root = Path.cwd()
for _ in range(5):
    if (_proj_root / "pipeline").is_dir() and (_proj_root / "notebooks").is_dir():
        if str(_proj_root) not in sys.path:
            sys.path.insert(0, str(_proj_root))
            print(f"  ✅ 项目根目录已加入 sys.path: {_proj_root}")
        break
    _proj_root = _proj_root.parent
else:
    # 兜底: 直接加 cwd
    if str(Path.cwd()) not in sys.path:
        sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from pipeline.data_pipeline import (
    find_project_root,
    sniff_raw_data_path,
    load_movies,
    load_data_snapshot,
    save_recall_results,
    compute_recall_at_k,
)

warnings.filterwarnings("ignore")

# ---------- 国内镜像 ----------
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# ---------- 路径配置 (自适应嗅探) ----------
# 使用 _proj_root (已通过 pipeline/ + notebooks/ 确认正确)，而非 find_project_root()
# 避免因 notebooks/data/raw/ 下的副本导致路径误判
BASE_DIR = _proj_root
RAW_DIR, RATINGS_PATH, MOVIES_PATH = sniff_raw_data_path(BASE_DIR)
RESULTS_DIR = BASE_DIR / "data" / "results"
SNAPSHOT_DIR = BASE_DIR / "data" / "snapshots"
CACHE_DIR = str(BASE_DIR / "models" / "hub")
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

print(f"✅ 项目根目录: {BASE_DIR}")
print(f"✅ 原始数据:   {MOVIES_PATH}")
print(f"✅ 快照目录:   {SNAPSHOT_DIR}")
print(f"✅ 结果保存:   {RESULTS_DIR}")

# ---------- 模型 & 检索超参数 ----------
MODEL_NAME = "Qwen/Qwen2.5-0.5B"   # 调试用小模型，可替换为 Qwen3-1.8B
PROJECTION_DIM = 128                # Embedding 维度
TOP_K = 50                          # 召回 Top-50
MAX_USER_HISTORY = 50               # 用户历史序列最大长度
BATCH_SIZE = 32                     # Embedding 批处理大小 (动态 padding)
POOL_STRATEGY = "last"              # 池化策略: last → 取最后一个非PAD token的隐状态
MAX_SEQ_LEN = 512                   # 模型最大上下文长度 (左截断从左侧丢弃)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- CUDA 诊断 (确认 GPU 可用性 & 型号) ---
if DEVICE.type == "cuda":
    print(f"  ✅ CUDA 可用! GPU: {torch.cuda.get_device_name(0)}")
    print(f"     显存总量:    {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"     当前占用:    {torch.cuda.memory_allocated(0) / 1024**2:.0f} MB")
    print(f"     缓存占用:    {torch.cuda.memory_reserved(0) / 1024**2:.0f} MB")
else:
    print("  ⚠️  CUDA 不可用! 将在 CPU 上运行，速度会非常慢。")
    print("     请检查: nvidia-smi / CUDA 版本 / PyTorch 是否安装 CUDA 版本")

print(f"\n🔧 配置:")
print(f"  模型: {MODEL_NAME}")
print(f"  设备: {DEVICE}")
print(f"  Embedding 维度: {PROJECTION_DIM}")
print(f"  召回 Top-K: {TOP_K}")
print(f"  Batch Size: {BATCH_SIZE} (动态 Padding)")
print(f"  用户历史最大长度: {MAX_USER_HISTORY}")
print(f"  池化策略: {POOL_STRATEGY} (最后一个非PAD token隐状态)")
print(f"  最大上下文: {MAX_SEQ_LEN} (左截断策略)")
print(f"  Item输入: [指令前缀] + 结构化KV文本 (无ID特征)")
print(f"  User输入: 历史记录 + 当前搜索 (左截断优先保留近期)")

✅ 项目根目录: /home/hjy/Videos/LLM-REC
✅ 原始数据:   /home/hjy/Videos/LLM-REC/data/raw/ml-1m/movies.dat
✅ 快照目录:   /home/hjy/Videos/LLM-REC/data/snapshots
✅ 结果保存:   /home/hjy/Videos/LLM-REC/data/results
  ✅ CUDA 可用! GPU: NVIDIA GeForce GTX 1060 6GB
     显存总量:    5.92 GB
     当前占用:    0 MB
     缓存占用:    0 MB

🔧 配置:
  模型: Qwen/Qwen2.5-0.5B
  设备: cuda
  Embedding 维度: 128
  召回 Top-K: 50
  Batch Size: 32 (动态 Padding)
  用户历史最大长度: 50
  池化策略: last_token (最后一个token隐状态)
  最大上下文: 512
  Item输入: [指令前缀] + 结构化KV (无ID特征)
  User输入: 历史记录 + 当前搜索 (左截断)


---
## 1. 加载共享数据快照

从 `data/snapshots/` 加载 ALS 笔记本保存的 Train/Test 数据快照。

**关键**: 
- 使用快照中的数据，而非重新加载原始数据重新划分
- Train Set → 用户历史记忆 (User Tower)
- Test Set → Ground Truth 评估

In [6]:
# ============================================================
# Cell 2: 加载共享数据快照
# ============================================================
print("=" * 60)
print("【1/6】加载共享数据快照...")
print("=" * 60)

try:
    # 尝试从快照加载 (与 ALS 笔记本共享完全相同的数据)
    train_df, test_df, movies = load_data_snapshot(SNAPSHOT_DIR, prefix="ml_1m")
    print(f"  ✅ 成功加载共享数据快照!")
except FileNotFoundError:
    # 回退: 自己加载原始数据并划分
    print("  ⚠️  未找到共享快照，将从原始数据创建...")
    from pipeline.data_pipeline import (
        load_ratings,
        chronological_split,
        save_data_snapshot,
    )
    ratings = load_ratings(RATINGS_PATH)
    train_df, test_df = chronological_split(ratings, strategy="loo", n_test=1)
    del ratings
    movies = load_movies(MOVIES_PATH)
    save_data_snapshot(train_df, test_df, movies, SNAPSHOT_DIR, prefix="ml_1m")
    print(f"  ✅ 已创建并保存数据快照!")

# --- 构建电影信息快速查找 ---
movie_info = movies.set_index("MovieID")[["Title", "Genres"]].to_dict("index")
all_movie_ids = sorted(movies["MovieID"].tolist())
movie_id_to_idx = {mid: i for i, mid in enumerate(all_movie_ids)}
idx_to_movie_id = {i: mid for mid, i in movie_id_to_idx.items()}

print(f"\n  总电影数: {len(all_movie_ids):,}")
print(f"  训练集记录: {len(train_df):,} (用户历史记忆)")
print(f"  测试集记录: {len(test_df):,} (Ground Truth)")
print(f"  训练集用户: {train_df['UserID'].nunique():,}")
print(f"  测试集用户: {test_df['UserID'].nunique():,}")

【1/6】加载共享数据快照...
  ✅ 数据快照加载成功:
     - 训练集: ml_1m_train.parquet (994,169 条)
     - 测试集: ml_1m_test.parquet (6,040 条)
     - 电影:   ml_1m_movies.parquet (3,883 部)
  ✅ 成功加载共享数据快照!

  总电影数: 3,883
  训练集记录: 994,169 (用户历史记忆)
  测试集记录: 6,040 (Ground Truth)
  训练集用户: 6,040
  测试集用户: 6,040


---
## 2. 加载 Qwen 模型 (共享双塔编码器)

使用 `QwenDualTowerRecall` 模型，共享 Qwen Backbone + 独立 User/Item 投影层。

In [8]:
# ============================================================
# Cell 3: 加载 Qwen 模型与 Tokenizer
# ============================================================
print("=" * 60)
print("【2/6】加载 Qwen 模型与 Tokenizer...")
print("=" * 60)

# --- 0. 版本诊断 (排查 JIT/类型注解兼容性问题) ---
print(f"  📦 PyTorch 版本: {torch.__version__}")
import transformers as _trf
print(f"  📦 Transformers 版本: {_trf.__version__}")
print(f"  📦 JIT 状态: {'禁用(已设置' if os.environ.get('PYTORCH_JIT', '1') == '0' else '启用(未设置'})")
del _trf

sys.path.insert(0, str(BASE_DIR))
import importlib
import models.qwen_dual_tower
importlib.reload(models.qwen_dual_tower)  # 强制重新加载，确保使用最新的 .py 文件
from models.qwen_dual_tower import QwenDualTowerRecall
from transformers import AutoTokenizer

# --- 3a. 加载 Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    cache_dir=CACHE_DIR,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("✅ Tokenizer 加载成功")
print(f"   PAD token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"   EOS token: {tokenizer.eos_token} (id={tokenizer.eos_token_id})")

# --- 3b. 确定精度: GPU 上优先用 bfloat16 (比 float16 更稳定) ---
#     float16/bf16 相比 float32: 显存减半、Tensor Cores 加速 ~2x
if DEVICE.type == "cuda":
    cap = torch.cuda.get_device_capability()
    if cap[0] >= 8:  # Ampere+ 架构 (A100, RTX 30xx/40xx)
        model_dtype = torch.bfloat16
        print(f"  ✅ GPU 架构支持 bfloat16 (sm_{cap[0]}{cap[1]}), 使用 bfloat16")
    else:  # Turing/Volta/Pascal 架构 (GTX 10xx, RTX 20xx, V100)
        model_dtype = torch.float16
        print(f"  ✅ GPU 架构 sm_{cap[0]}{cap[1]}, 使用 float16")
else:
    model_dtype = torch.float32  # CPU 不支持 half 加速

# --- 3c. 加载双塔模型 (半精度 + GPU) ---
#     [JIT 兼容性] PyTorch 较旧版本的 JIT schema 解析器无法正确处理
#     Qwen2.5 的 @torch.jit.script 字符串类型注解。模型自行通过
#     猴子补丁 torch.jit.script 为 lambda f: f 来绕过此 Bug。
#     [池化策略] last → 取最后一个非PAD token的隐状态。
#     配合左截断策略，确保模型重点关注最新的输入信息（近期行为+当前搜索意图）。
model = QwenDualTowerRecall(
    model_name=MODEL_NAME,
    projection_dim=PROJECTION_DIM,
    pool_strategy=POOL_STRATEGY,
    torch_dtype=model_dtype,
).to(DEVICE)

model.eval()
print(f"✅ 模型加载成功 (设备={DEVICE}, 精度={model_dtype})")

# --- 3d. 打印 GPU 显存使用情况 ---
if DEVICE.type == "cuda":
    print(f"   显存占用: {torch.cuda.memory_allocated(0) / 1024**2:.0f} MB")
    print(f"   显存缓存: {torch.cuda.memory_reserved(0) / 1024**2:.0f} MB")

【2/6】加载 Qwen 模型与 Tokenizer...
  📦 PyTorch 版本: 2.4.1+cu121
  📦 Transformers 版本: 5.8.0
  📦 JIT 状态: 禁用(已设置)
✅ Tokenizer 加载成功
   PAD token: <|endoftext|> (id=151643)
   EOS token: <|endoftext|> (id=151643)
  ✅ GPU 架构 sm_61, 使用 float16


AttributeError: 'QwenDualTowerRecall' object has no attribute 'encoder'

---
## 3. Item Tower: 提取全部电影 Embedding → FAISS 索引

**输入格式**: `简短任务指令前缀 + 物品Key-Value结构化属性文本`
- 不拼接 ID 类特征（如 MovieID），避免引入无效噪声
- 统一添加语义指令前缀，让模型精准识别当前为「物品语义编码」任务
- 核心：完成物品静态属性的结构化语义表达

**性能优化**: 使用 `padding=True` (动态 Padding) 而非 `max_length` 硬截断。
Batch 内只 padding 到当前 batch 最长序列，节省 ~50% 显存。

In [ ]:
# ============================================================
# Cell 4: Item Tower — 提取全部电影 Embedding
# ============================================================
print("=" * 60)
print(f"【3/6】Item Tower: 提取全部 {len(all_movie_ids)} 部电影 Embedding...")
print("=" * 60)

# --- 4a. 构造电影描述文本 (任务指令前缀 + 结构化KV) ---
# [设计原则]
# 1. 统一添加语义指令前缀 → 让模型精准识别编码任务，而非做生成
# 2. 不使用ID特征 → 避免过拟合ID噪声，聚焦静态属性语义
# 3. Key-Value结构化 → title=X genre=Y 显式表达属性边界
ITEM_INSTRUCTION = "Encode the following item attributes into a semantic vector:"
item_texts = []
for mid in all_movie_ids:
    info = movie_info[mid]
    # 结构化 KV 文本: 不拼接 MovieID, 使用统一指令前缀
    kv_text = f"title={info['Title']} genre={info['Genres']}"
    text = f"{ITEM_INSTRUCTION} {kv_text}"
    item_texts.append(text)

print(f"  电影描述样本: {item_texts[0][:120]}...")
print(f"  总电影数: {len(item_texts):,}")

# --- 4b. 批量提取 Item Embedding (动态 Padding) ---
# [GPU 防护] 确保模型在正确的设备上 (即使独立执行此 Cell)
model = model.to(DEVICE)
model.eval()
first_param_device = next(model.parameters()).device
print(f"  🖥️ 模型设备: {first_param_device}  (期望: {DEVICE})")
assert first_param_device.type == DEVICE.type, \
    f"模型在 {first_param_device} 但期望在 {DEVICE}，请先执行 Cell 3!"

# ── 收集阶段: 在 GPU 上维持 PyTorch 张量 ──
all_item_embeddings: list[torch.Tensor] = []
for i in tqdm(range(0, len(item_texts), BATCH_SIZE), desc="Item Tower (动态Padding)"):
    batch_texts = item_texts[i:i + BATCH_SIZE]

    # 核心优化: padding=True → 动态 Padding
    # 只填充到当前 batch 最长序列，不浪费计算资源
    enc = tokenizer(
        batch_texts,
        padding=True,          # 动态 Padding
        truncation=True,       # 安全截断，防止极端长文本 OOM
        max_length=128,        # 安全上限 (电影文本通常 < 64 tokens)
        return_tensors="pt",
    )

    with torch.no_grad():
        emb = model.get_item_embedding(
            enc["input_ids"].to(DEVICE),
            enc["attention_mask"].to(DEVICE),
        )
    # 保持 GPU 张量，后续统一转换
    all_item_embeddings.append(emb)

# ── 转换阶段: 一次性转为 CPU float32 numpy (供 FAISS 使用) ──
# FAISS 的 C++ 后端 (normalize_L2, index.add, index.search) 只接受
# CPU 上的 float32 numpy 数组，不接受 float16/bfloat16 或 PyTorch 张量。
item_embeddings = torch.cat(all_item_embeddings, dim=0).cpu().float().numpy()
del all_item_embeddings, item_texts

print(f"\n  ✅ Item Embedding 矩阵形状: {item_embeddings.shape}")
print(f"     数据类型: {item_embeddings.dtype}")
print(f"     内存占用: {item_embeddings.nbytes / 1024**2:.2f} MB")
assert item_embeddings.dtype == np.float32, \
    f"FAISS 需要 float32, 实际为 {item_embeddings.dtype}"

In [ ]:
# ============================================================
# Cell 5: 构建 FAISS 索引
# ============================================================
print("=" * 60)
print("【4/6】构建 FAISS 索引...")
print("=" * 60)

import faiss

# ── 前置断言: FAISS 只接受 CPU float32 numpy ──
assert isinstance(item_embeddings, np.ndarray), \
    f"FAISS 需要 numpy.ndarray, 实际为 {type(item_embeddings)}"
assert item_embeddings.dtype == np.float32, \
    f"FAISS 需要 float32, 实际为 {item_embeddings.dtype}"
assert item_embeddings.data.c_contiguous, \
    "FAISS 需要 C-contiguous 数组"

print(f"  📊 输入验证通过: {item_embeddings.shape}, {item_embeddings.dtype}, contiguous")
print(f"     内存占用: {item_embeddings.nbytes / 1024**2:.2f} MB")

# --- 5a. L2 归一化 (in-place, FAISS C++ 后端) ---
faiss.normalize_L2(item_embeddings)
print("  ✅ Item Embedding L2 归一化完成")

# --- 5b. 构建 Flat IP (内积) 索引 ---
dimension = item_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(item_embeddings)
print(f"  ✅ FAISS 索引构建完成")
print(f"     索引中向量数: {index.ntotal}")
print(f"     向量维度: {dimension}")

---
## 4. User Tower: 用户 Embedding 批量生成 & FAISS 检索

**核心架构调整 — 时序信息与实时意图权重优先级**：
- 用户侧文本拼接顺序固定为：`用户历史记录 + 当前搜索内容`
- 当前搜索内容在序列末尾（Last Token 位置），配合 Last Token Pooling 拥有更高权重
- 若拼接后文本超出模型最大上下文上限，从**左侧截断**，优先保留近期行为与当前搜索意图
- 严格使用 Train Set (历史记忆) 构建用户画像，无数据泄露

**输入格式**:
```
[用户历史记录: title=... genre=... | title=... genre=...] + [当前搜索内容]
                                           ▲                                        
                           左截断从左侧丢弃 ---→     ←--- 保留右侧 (近期+当前意图)    
                                           └── Last Token → Embedding                
```

**修复关键 Bug**: 
- 原始代码仅对 82 个用户进行了召回
- 现在对所有 ~6040 个测试用户进行全量召回

In [ ]:
# ============================================================
# Cell 6: User Tower — 批量生成 User Embedding & FAISS 检索
# ============================================================
print("=" * 60)
print("【5/6】User Tower: 批量生成 User Embedding & FAISS 检索...")
print("=" * 60)

# ==================== 6a. 构建用户历史文本 ====================
# 严格使用 Train Set (历史记忆) 构建用户画像
# 数据划分由 shared pipeline 保证，与 ALS 笔记本完全一致
print("\n  📋 构建用户历史文本 (严格使用 Train Set)...")
print(f"     Train Set 记录数: {len(train_df):,}")
print(f"     Train Set 用户数: {train_df['UserID'].nunique():,}")

# [架构] 当前搜索内容 (ML-1M 场景下为空; 真实系统可替换为用户实时搜索查询)
# 拼接顺序固定为: 用户历史记录 + 当前搜索内容
# 当 CURRENT_SEARCH_CONTENT 非空时, 其在序列末尾, 配合 Last Token Pooling 获得更高权重
CURRENT_SEARCH_CONTENT = ""

# 按 UserID + Timestamp 排序 (确保时间顺序正确)
train_df = train_df.sort_values(["UserID", "Timestamp"])

user_history_texts = {}
user_history_count = {}

for uid, group in train_df.groupby("UserID"):
    # 取最近的 MAX_USER_HISTORY 部电影 (tail = 最新)
    recent_movies = group.tail(MAX_USER_HISTORY)["MovieID"].values
    history_parts = []

    for mid in recent_movies:
        if mid in movie_info:
            info = movie_info[mid]
            history_parts.append(f"title={info['Title']} genre={info['Genres']}")

    if history_parts:
        # [架构] 最终文本 = 用户历史记录 + 当前搜索内容
        # 历史记录在前 (左), 当前搜索在后 (右 → 靠近 Last Token)
        history_str = " | ".join(history_parts)
        full_text = history_str + CURRENT_SEARCH_CONTENT
        user_history_texts[int(uid)] = full_text
        user_history_count[int(uid)] = len(history_parts)

print(f"     ✅ 构建了 {len(user_history_texts):,} 个用户的 history 文本")
print(f"     📊 用户历史长度统计: "
      f"min={min(user_history_count.values())}, "
      f"max={max(user_history_count.values())}, "
      f"mean={np.mean(list(user_history_count.values())):.1f}")
if CURRENT_SEARCH_CONTENT:
    print(f"     🔍 当前搜索内容: \"{CURRENT_SEARCH_CONTENT[:60]}\" (序列末尾 → Last Token 高权重)")
else:
    print(f"     ℹ️  当前搜索内容为空 (仅使用历史记录, 序列末尾为最后一部电影)")

# ==================== 6b. 构建测试用户列表 ====================
# 只选择: 在测试集中有 ground truth + 有历史记录的测试用户
# 构建 Test Set Ground Truth
from pipeline.data_pipeline import build_test_ground_truth
test_user_groups = build_test_ground_truth(test_df)

valid_test_users = sorted([
    uid for uid in test_user_groups
    if uid in user_history_texts and len(test_user_groups[uid]) > 0
])
print(f"\n  🎯 测试用户数 (有 GT + 有历史): {len(valid_test_users):,}")
print(f"     原始测试集用户: {len(test_user_groups):,}")
print(f"     有历史记录的用户: {len(user_history_texts):,}")

del train_df  # 释放内存

# ==================== 6c. 批量生成 User Embedding ====================
print(f"\n  🚀 开始批量生成 User Embedding (Batch Size={BATCH_SIZE}, 动态Padding + 左截断)...")
print(f"     总共 {len(valid_test_users)} 个用户，{len(valid_test_users)//BATCH_SIZE + 1} 个 Batch")
print(f"     最大上下文: {MAX_SEQ_LEN} tokens (超长从左侧截断)")

# [GPU 防护] 确保模型在正确的设备上 (即使独立执行此 Cell)
model = model.to(DEVICE)
model.eval()
first_param_device = next(model.parameters()).device
print(f"  🖥️ 模型设备: {first_param_device}  (期望: {DEVICE})")
assert first_param_device.type == DEVICE.type, \
    f"⚠️ 模型在 {first_param_device} 但期望在 {DEVICE}! 请先执行 Cell 3！"

# ── 收集阶段: 在 GPU 上维持 PyTorch 张量 ──
all_user_embeddings: list[torch.Tensor] = []
all_user_ids: list[int] = []

# [关键修复] 设置 tokenizer 的 truncation_side 为 "left"
# - Tokenizer 内置左截断：超长文本从左侧丢弃，保留右侧（近期行为 + 当前搜索意图）
# - 配合 Last Token Pooling，模型重点关注序列末尾的近期信息
# - 比手动截断 tensor 更可靠，能正确处理不同长度的序列
tokenizer.truncation_side = "left"

# 按 Batch 分组处理
for start_idx in tqdm(range(0, len(valid_test_users), BATCH_SIZE), desc="User Tower (左截断+动态Padding)"):
    batch_users = valid_test_users[start_idx:start_idx + BATCH_SIZE]

    # 构造这一批用户的完整文本: 历史记录 + 当前搜索内容
    batch_texts: list[str] = []
    for uid in batch_users:
        try:
            text = user_history_texts[uid]
            # 输入验证：确保是字符串
            if not isinstance(text, str):
                text = str(text) if text is not None else ""
            batch_texts.append(text)
        except KeyError:
            # 异常处理：跳过该用户（无历史记录）
            batch_texts.append("")
            print(f"  ⚠️  用户 {uid} 无历史文本，返回空向量")

    # ── [核心] 使用 Tokenizer 内置的左截断 + 动态 Padding ──
    # Tokenizer 的 padding=True 会根据 batch 内最长序列动态填充；
    # truncation=True + truncation_side="left" 实现从左侧丢弃超长部分。
    enc = tokenizer(
        batch_texts,
        padding=True,          # 动态 Padding（兼容不同长度序列）
        truncation=True,       # 超长截断
        max_length=MAX_SEQ_LEN,# 以右侧为锚点截断（受 truncation_side 控制）
        return_tensors="pt",
    )

    with torch.no_grad():
        emb = model.get_user_embedding(
            enc["input_ids"].to(DEVICE),
            enc["attention_mask"].to(DEVICE),
        )

    # 保持 GPU 张量，后续统一转换
    all_user_embeddings.append(emb)
    all_user_ids.extend(batch_users)

# ── 转换阶段: 一次性转为 CPU float32 numpy (供 FAISS 使用) ──
user_embeddings = torch.cat(all_user_embeddings, dim=0).cpu().float().numpy()
del all_user_embeddings, user_history_texts

# [还原] 恢复 tokenizer truncation_side 为默认值 (right)，
# 避免影响后续 Cell 中 Item Tower 或其他编码任务的行为
tokenizer.truncation_side = "right"

print(f"\n  ✅ User Embedding 矩阵形状: {user_embeddings.shape}")
print(f"     数据类型: {user_embeddings.dtype}")
print(f"     用户数: {user_embeddings.shape[0]}")
assert user_embeddings.dtype == np.float32, \
    f"FAISS 需要 float32, 实际为 {user_embeddings.dtype}"

# ==================== 6d. FAISS 批量检索 Top-50 ====================
print(f"\n  🔍 FAISS 批量检索 Top-{TOP_K}...")

# ── 前置断言: FAISS 只接受 CPU float32 numpy ──
assert isinstance(user_embeddings, np.ndarray), \
    f"FAISS 需要 numpy.ndarray, 实际为 {type(user_embeddings)}"
assert user_embeddings.dtype == np.float32, \
    f"FAISS 需要 float32, 实际为 {user_embeddings.dtype}"
assert user_embeddings.data.c_contiguous, \
    "FAISS 需要 C-contiguous 数组"
print(f"     输入验证通过: {user_embeddings.shape}, {user_embeddings.dtype}, contiguous")

# User Embedding L2 归一化 (in-place, FAISS C++ 后端)
faiss.normalize_L2(user_embeddings)

# 批量检索: 一次调用检索全部用户的 Top-K
# index.search 返回 (distances, indices), 形状: [n_users, top_k]
D, I = index.search(user_embeddings, TOP_K)

# 将 FAISS 的 0-based 索引映射回原始 MovieID
llm_recall_dict = {}
for user_pos, uid in enumerate(all_user_ids):
    # I[user_pos] 是 0-based 的 FAISS 索引
    rec_movie_ids = [idx_to_movie_id[faiss_idx] for faiss_idx in I[user_pos]]
    llm_recall_dict[int(uid)] = [int(iid) for iid in rec_movie_ids]

print(f"  ✅ FAISS 检索完成!")
print(f"     覆盖用户数: {len(llm_recall_dict):,}")
print(f"     每个用户召回数: {TOP_K}")

del user_embeddings, all_user_ids

---
## 5. 持久化保存 LLM 召回结果 & 评估

保存格式: `{user_id: [item_id1, item_id2, ..., item_id50]}`

评估: 计算 Recall@K 并与 ALS 结果对比。

In [ ]:
# ============================================================
# Cell 7: 保存 LLM 召回结果 & 评估
# ============================================================
print("=" * 60)
print("【6/6】保存 LLM 召回结果...")
print("=" * 60)

llm_output_path = RESULTS_DIR / "llm_recall_ml_1m.json"
save_recall_results(llm_recall_dict, llm_output_path, "Qwen Dual-Tower", TOP_K)

# --- 评估 Recall@K ---
print(f"\n  📊 评估 Qwen 双塔召回性能...")
mean_recall = compute_recall_at_k(llm_recall_dict, test_user_groups, TOP_K)

# --- 输出数据一致性验证 ---
print(f"\n{'=' * 60}")
print(f"✅ Qwen 双塔语义召回模块重构完成!")
print(f"{'=' * 60}")
print(f"\n  📊 召回结果摘要:")
print(f"     用户数:          {len(llm_recall_dict):,}")
print(f"     Top-K:           {TOP_K}")
print(f"     存储路径:        {llm_output_path}")
print(f"     Recall@{TOP_K}: {mean_recall:.4f} ({mean_recall*100:.2f}%)")

print(f"\n  🔄 数据一致性验证:")
print(f"     ALS 结果: data/results/als_recall_ml_1m.json")
print(f"     LLM 结果: data/results/llm_recall_ml_1m.json")
print(f"     两份结果的 user_id 集合由共享的快照保证完全一致")

In [ ]:
# ============================================================
# Cell 8: (可选) 交叉验证 ALS vs Qwen 用户集一致性
# ============================================================
print("=" * 60)
print("交叉验证: ALS vs Qwen 用户集一致性")
print("=" * 60)

# 加载 ALS 结果
als_path = RESULTS_DIR / "als_recall_ml_1m.json"
if als_path.exists():
    with open(als_path, "r") as f:
        als_data = json.load(f)

    als_users = set(int(k) for k in als_data.keys())
    llm_users = set(int(k) for k in llm_recall_dict.keys())

    overlap = als_users & llm_users
    als_only = als_users - llm_users
    llm_only = llm_users - als_users

    print(f"\n  📊 用户集对比:")
    print(f"     ALS 用户数:  {len(als_users):,}")
    print(f"     Qwen 用户数: {len(llm_users):,}")
    print(f"     交集:        {len(overlap):,}")
    print(f"     ALS 独有:    {len(als_only):,}")
    print(f"     Qwen 独有:   {len(llm_only):,}")

    if len(als_only) == 0 and len(llm_only) == 0:
        print(f"\n  ✅ 完全一致! 两个模块的用户 ID 集合 100% 对齐!")
    else:
        print(f"\n  ⚠️  存在差异，请检查数据划分逻辑")
else:
    print(f"\n  ⚠️  ALS 结果文件不存在，跳过交叉验证")
    print(f"     请先运行 03_baseline_ALS_recall.ipynb")